In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

In [2]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
from shutil import copytree, ignore_patterns

# Define dataset path
dataset_path = "Dataset"

# Get all class directories
classes = sorted([d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))])
print(f"Total classes found: {len(classes)}")
print(f"Classes: {classes}")

Total classes found: 23
Classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']


In [3]:
# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2  # 80% train, 20% validation
)

# Data preprocessing for testing
test_datagen = ImageDataGenerator(rescale=1./255)

In [4]:
# Load data directly from Dataset folder
dataset_path = "Dataset"

# Training data with augmentation (80%)
train_data = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    subset='training'  # 80% for training
)

# Validation data (20%)
valid_data = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    subset='validation'  # 20% for validation
)

print(f"Total classes: {len(train_data.class_indices)}")
print(f"Classes: {list(train_data.class_indices.keys())}")
print(f"Training samples: {train_data.samples}")
print(f"Validation samples: {valid_data.samples}")

Found 28589 images belonging to 23 classes.
Found 7136 images belonging to 23 classes.
Total classes: 23
Classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']
Training samples: 28589
Validation samples: 7136


In [6]:
model = Sequential()

# Conv Layer 1
model.add(Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)))
model.add(MaxPooling2D(2,2))

# Conv Layer 2
model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D(2,2))

# Conv Layer 3
model.add(Conv2D(128, (3,3), activation='relu'))
model.add(MaxPooling2D(2,2))

# Conv Layer 4
model.add(Conv2D(256, (3,3), activation='relu'))
model.add(MaxPooling2D(2,2))

# Flatten
model.add(Flatten())

# Fully Connected
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

# Output Layer - 23 classes for your dataset
num_classes = len(classes)
model.add(Dense(num_classes, activation='softmax'))

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f"Model built for {num_classes} plant disease classes")
model.summary()

Model built for 23 plant disease classes
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 126, 126, 32)      896       
                                                                 
 max_pooling2d (MaxPooling2  (None, 63, 63, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 61, 61, 64)        18496     
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 30, 30, 64)        0         
 g2D)                                                            
                                                                 
 conv2d_2 (Conv2D)           (None, 28, 28, 128)       73856     
                                                                 
 max_pooling2d_

In [ ]:
history = model.fit(
    train_data,
    epochs=3,
    validation_data=valid_data
)

Epoch 1/3
894/894 [==============================] - 965s 1s/step - loss: 2.3063 - accuracy: 0.2975 - val_loss: 1.2284 - val_accuracy: 0.6149
Epoch 2/3
894/894 [==============================] - 770s 861ms/step - loss: 1.2865 - accuracy: 0.6007 - val_loss: 0.7125 - val_accuracy: 0.7712
Epoch 3/3
894/894 [==============================] - ETA: 0s - loss: 0.8945 - accuracy: 0.7192

In [ ]:
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')

plt.title("CNN Accuracy Graph")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

def upload_and_predict():
    """
    Upload image widget for prediction
    """
    uploader = widgets.FileUpload(
        accept='image/*',
        multiple=False,
        description='Choose Image'
    )
    
    output = widgets.Output()
    
    def on_upload_change(change):
        with output:
            output.clear_output()
            if uploader.value:
                uploaded_file = list(uploader.value.values())[0]
                
                # Save temporarily and predict
                with open('temp_uploaded.jpg', 'wb') as f:
                    f.write(uploaded_file['content'])
                
                print("🔍 Analyzing image...")
                result = visualize_prediction('temp_uploaded.jpg')
                
                print("\n" + "="*50)
                print("📊 PREDICTION RESULTS")
                print("="*50)
                print(f"Predicted Disease: {result['predicted_disease']}")
                print(f"Confidence: {result['confidence_percentage']}")
                print("\nTop 3 Predictions:")
                for i, pred in enumerate(result['top_3_predictions'], 1):
                    print(f"  {i}. {pred['disease']}: {pred['confidence']*100:.2f}%")
                print("="*50)
    
    uploader.observe(on_upload_change, names='value')
    
    display(HTML("<h3>📤 Upload Plant Image for Disease Detection</h3>"))
    display(uploader)
    display(output)

# Create the upload interface
upload_and_predict()